# 08 — R1 CIFAR100/K5, seed44,160 epochs, d8000

Train base mới cho seed44; class order, khởi tạo backbone/augmentation và FE đều theo seed này. Không dùng checkpoint seed42 hoặc seed khác. Giữ nguyên recipe của seed42 đã nghiệm thu: gamma0.1,50 base+5×10 lớp, LR80/120, d8000.

GPU T4/Internet; không cần Input. CHECKPOINT=None, RESUME_CHECKPOINT=None, STOP_AFTER_EPOCH=None. API dùng timeout7200 giây. Seed42 đo khoảng53,78 phút numeric; đây là tham chiếu dự trù cho phiên này, không bảo đảm runtime bằng nhau.

W&B ghi offline vào Output, sync đúng payload trên máy đã đăng nhập trước khi công bố. Giữ benchmark_seed44_ready.json, checkpoint/state/progress/results và log offline cùng version/attempt. Source bundle giống notebook06; không sửa source bên trong bundle.

Chỉ sau khi đủ cả42/43/44 mới tổng hợp mean±sample_SD (ddof=1). Tạo notebook này không tự gửi job GPU.


In [ ]:
import os, sys, json, base64, hashlib, io, tempfile, zipfile
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
ACTION = "benchmark"  # Preset R1:160 epochs, d8000, K5, seed44.
CHECKPOINT = None  # Chỉ reuse base160 epochs đã hoàn tất, cùng config.
RESUME_CHECKPOINT = None  # Chỉ checkpoint v2 đang dở với target160 epochs.
STOP_AFTER_EPOCH = None  # Ví dụ1 để kiểm chạy ngắt; giữ None khi chạy tới target.
LOG_WANDB = False  # Enable secret WANDB_API_KEY trên Kaggle.
PROFILE_DIMENSION = 8000
RUN_INCREMENTAL_PHASES = 5
BUNDLE_SHA256 = '5851fa25143689342edabe9ab112635395ed2eae4f19fe95c446eaf96b6ca19c'
candidates = (Path.cwd(), Path.cwd().parent)
PROJECT_ROOT = next((p for p in candidates if (p / "src/profile_analytic.py").is_file()), None)
if PROJECT_ROOT is None:
    data = base64.b64decode('...')
    assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256
    parent = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(tempfile.gettempdir())
    PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="acil-benchmark-source-", dir=parent))
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        for name in archive.namelist():
            assert (PROJECT_ROOT / name).resolve().is_relative_to(PROJECT_ROOT.resolve())
        archive.extractall(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Source:", PROJECT_ROOT)
print("Embedded bundle SHA-256:", BUNDLE_SHA256)


## Dependency — giữ PyTorch hiện có của Kaggle

In [ ]:
import subprocess, importlib.util, importlib.metadata
pins = {"numpy":"2.1.3", "huggingface_hub":"1.10.1", "pyarrow":"23.0.1",
        "Pillow":"11.1.0", "threadpoolctl":"3.6.0", "wandb":"0.28.1",
        "filelock":"4.2.6", "scipy":"1.17.0", "psutil":"7.2.2"}
missing = []
for package, version in pins.items():
    try: current = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: current = None
    if current != version: missing.append(f"{package}=={version}")
if missing: subprocess.run([sys.executable,"-m","pip","install","-q",*missing],check=True)
if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable,"-m","pip","install","torch==2.11.0"],check=True)
import torch, numpy as np
if np.__version__ != "2.1.3": raise RuntimeError("Restart kernel sau khi đổi NumPy rồi Run All")
print("NumPy:",np.__version__,"Torch:",torch.__version__,"CUDA:",torch.cuda.is_available())


## Tests — xuất bằng chứng resume CUDA vào JSON

In [ ]:
from datetime import datetime, timezone
import uuid
working = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else PROJECT_ROOT / "reports/local"
proof_dir = working / "benchmark_checks" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"-"+uuid.uuid4().hex[:8])
proof_dir.mkdir(parents=True)
proof_path = proof_dir / "tests.json"
checked = subprocess.run([sys.executable,"-B","-X","utf8","-c",'import hashlib, json, platform, sys, unittest\nfrom pathlib import Path\nimport numpy as np\nimport torch\n\nclass RecordedResult(unittest.TextTestResult):\n    def __init__(self,*args,**kwargs):\n        super().__init__(*args,**kwargs);self.outcomes={}\n    def addSuccess(self,test):\n        super().addSuccess(test);self.outcomes[test.id()]="passed"\n    def addSkip(self,test,reason):\n        super().addSkip(test,reason);self.outcomes[test.id()]="skipped: "+reason\n    def addFailure(self,test,error):\n        super().addFailure(test,error);self.outcomes[test.id()]="failed"\n    def addError(self,test,error):\n        super().addError(test,error);self.outcomes[test.id()]="error"\n\nsuite=unittest.defaultTestLoader.discover("tests")\nresult=unittest.TextTestRunner(verbosity=2,resultclass=RecordedResult).run(suite)\ncuda_id="test_resume.ResumeTests.test_cuda_restart_matches_continuous_exactly"\npayload={"status":"complete" if result.wasSuccessful() else "failed",\n "verification_passed":result.wasSuccessful(),"tests_run":result.testsRun,"skipped":len(result.skipped),\n "failures":len(result.failures),"errors":len(result.errors),"outcomes":result.outcomes,\n "cuda_available":torch.cuda.is_available(),"cuda_resume_test_id":cuda_id,\n "cuda_resume_passed":result.outcomes.get(cuda_id)=="passed",\n "environment":{"python":platform.python_version(),"numpy":np.__version__,"torch":str(torch.__version__),\n "cuda_device":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},\n "bundle_sha256":sys.argv[2]}\nPath(sys.argv[1]).write_text(json.dumps(payload,ensure_ascii=False,indent=2)+"\\n",encoding="utf-8",newline="\\n")\nsys.exit(0 if result.wasSuccessful() else 1)\n',str(proof_path),BUNDLE_SHA256],
                         cwd=PROJECT_ROOT,capture_output=True,text=True,encoding="utf-8")
(proof_dir/"stdout.log").write_text(checked.stdout,encoding="utf-8",newline="\n")
(proof_dir/"stderr.log").write_text(checked.stderr,encoding="utf-8",newline="\n")
print(checked.stdout)
print(checked.stderr)
checked.check_returncode()
TEST_PROOF = json.loads(proof_path.read_text(encoding="utf-8"))
print("CUDA resume passed:",TEST_PROOF["cuda_resume_passed"],"Proof:",proof_path)


## Chạy R1 — yêu cầu CUDA resume đạt, giữ đúng protocol160/d8000

In [ ]:
import time
from src.cifar_smoke import run
from src.profile_analytic import MemoryMonitor
from src.reporting import save_json
import wandb

ready_path=working/"benchmark_seed44_ready.json"
save_json({"status":"running","verification_passed":False,"tests":TEST_PROOF},ready_path)
try:
    if ACTION!="benchmark" or not TEST_PROOF["cuda_resume_passed"] or not torch.cuda.is_available():
        raise RuntimeError("R1 cần GPU và test resume CUDA đạt.")
    if CHECKPOINT is not None and RESUME_CHECKPOINT is not None:
        raise ValueError("Chọn reuse hoặc resume, không chọn đồng thời.")
    with MemoryMonitor("cuda") as resources:
        report=run(None,config_path=PROJECT_ROOT/"configs/cifar100_benchmark_k5.json",
            config_overrides={"experiment":"ACIL-R1-C100-K5","purpose":"benchmark","seed":44,"device":"cuda",
                              "torch_threads":min(4,os.cpu_count() or 2),"run_incremental_phases":5},
            data_dir=working/"cifar-data",output_dir=working/"benchmark_seed44",
            reuse_checkpoint=CHECKPOINT,resume_checkpoint=RESUME_CHECKPOINT,stop_after_epoch=STOP_AFTER_EPOCH,
            log_wandb=False)
    assert report["status"]=="complete" and report["verification_passed"]
    assert report["config"]["epochs"]==160 and report["config"]["expansion_dim"]==8000
    assert len(report["training_history"])==160 and len(report["results"])==6
    assert report["results"][-1]["weights_shape"]==[8000,100]
    assert report["results"][-1]["test_samples_all_seen"]==10000
    assert report["joint_reference_status"]=="not_run" and report["streaming"]["stored_training_samples"]==0
    assert report["metric_summary"]["complete_protocol"] is True
    assert report["metric_summary"]["average_incremental_accuracy_pct"] is not None
    checkpoint=torch.load(Path(report["output_dir"])/"base_checkpoint.pt",map_location="cpu",weights_only=True)
    assert checkpoint["checkpoint_format_version"]==2 and checkpoint["training_complete"]
    assert checkpoint["completed_epoch"]==checkpoint["target_epochs"]==160
    bp_seconds=sum(row["seconds"] for row in report["training_history"])
    with wandb.init(entity="hsangg-2006",project="acil-reproduction",mode="offline",dir=str(working),
                    job_type="benchmark",tags=["R1","seed44","K5","d8000","offline-to-sync"],
                    config={**report["config"],"source_sha256":report["provenance"]["source_sha256"],
                            "kaggle_attempt_id":report["attempt_id"],"logging_origin":"Kaggle offline; sync after run"}) as offline_run:
        for row in report["results"]:
            offline_run.log({k:v for k,v in row.items() if isinstance(v,(int,float,bool))},step=row["phase"])
        offline_run.summary.update(report["metric_summary"])
        offline_run.summary.update({"verification_passed":report["verification_passed"],
            "cuda_resume_passed":True,"bp160_seconds":bp_seconds,
            "profile/peak_cpu_rss_bytes":resources.result["peak_cpu_rss_bytes"],
            "profile/peak_cuda_allocated_bytes":resources.result["peak_cuda_allocated_bytes"]})
        offline_id=offline_run.id
        offline_directory=str(Path(offline_run.dir).parent)
    ready={"status":"complete","verification_passed":True,"scope":"R1 seed44 full protocol; single seed, not three-seed aggregate",
           "attempt_id":report["attempt_id"],"benchmark":report,"tests":TEST_PROOF,
           "memory":{"scope":"training+analytic; notebook tests and offline logging excluded",**resources.result},
           "planning_budget":{'scope': 'Planning estimate for R1 seed44; transferred from seed42 profiles, not measured runtime of this seed', 'input_attempt_id': '20261104T173456176348Z-f3b67e52a9c7', 'bp160_linear_seconds': 2435.430647551999, 'analytic_linear_seconds': 802.2632924949988, 'feature_evaluation_overhead_scaled_seconds': 54.278414011707405, 'startup_test_allowance_seconds': 120, 'large_state_io_allowance_seconds': 120, 'estimated_total_seconds': 3531.972354058705, 'minimum_window_with_30_percent_remaining_seconds': 5045.674791512436, 'planned_timeout_seconds': 7200, 'planned_window_spare_fraction': 0.5094482841585132, 'user_reported_gpu_quota_hours': 26, 'quota_source_date': '2026-11-04', 'remaining_quota_note': 'User report, not live account reading; prior short runs consume some quota.', 'gpu_resume_verified': True, 'numerical_bp5_verified': True, 'tracking_status': 'synced', 'profile_reference_seed': 42, 'completed_seed42_numeric_pipeline_seconds': 3226.82625715, 'estimate_transfer_note': 'Same160/d8000/K5 recipe and2-hour timeout; actual timing recorded per seed'},
           "tracking":{"status":"offline_pending_sync","offline_run_id":offline_id,"offline_directory":offline_directory}}
    save_json(ready,Path(report["output_dir"])/"benchmark_seed44_ready.json")
    save_json(ready,ready_path)
except BaseException as error:
    save_json({"status":"failed","verification_passed":False,"error_type":type(error).__name__,"tests":TEST_PROOF},ready_path)
    raise


## Kết quả đủ100 lớp và log offline của chính lượt này

In [ ]:
assert ready["status"]=="complete" and ready["verification_passed"]
print("R1 SEED44 COMPLETE",ready_path)
print("Metrics:",report["metric_summary"])
print("Memory:",ready["memory"])
print("Offline tracking:",ready["tracking"])
print("Attempt:",report["attempt_id"],"Artifacts:",report["output_dir"])
print("Giữ Output/checkpoint và sync W&B trước khi công bố benchmark.")
